# Task 2: YOLOv8 training and evaluation (garbage detection)
Final run: yolov8s, imgsz 416 (Task 1b), default augmentation (Task 1b), max_det 400, seed 42. Same split as `prepare_data.py` (stratified 70/15/15).

In [ ]:
!pip install -q ultralytics
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
%%writefile prepare_data.py
import argparse
import os
import re
import shutil
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split


# The 6 classes in the same order as the IDs in the label files (0-5)
CLASSES = ["biodegradable", "cardboard", "glass", "metal", "paper", "plastic"]


def category(filename):
    """re is Python’s tool for finding text patterns.
    r"[a-z]+" means “one or more lowercase letters”.
    Return the waste category from a file name.
    Example: 'glass12_jpg.rf.abc.jpg' -> 'glass'
    """
    return re.match(r"[a-z]+", filename).group()


def label_name(image_name):
    """Return the label file that belongs to an image.
    Example: 'glass12.jpg' -> 'glass12.txt'
    """
    return os.path.splitext(image_name)[0] + ".txt"

def copy_dataset(src, dst, workers=64):
    """Copy images and labels to a fast, writable folder.
    Files that already exist are skipped.
    """
    jobs = []
    for sub in ("images", "labels"):
        os.makedirs(os.path.join(dst, sub), exist_ok=True)
        for name in os.listdir(os.path.join(src, sub)):
            target = os.path.join(dst, sub, name)
            if not os.path.exists(target):
                jobs.append((os.path.join(src, sub, name), target))

    print(f"[copy] files to copy: {len(jobs)}")
    with ThreadPoolExecutor(max_workers=workers) as pool:
        list(pool.map(lambda pair: shutil.copyfile(*pair), jobs))
    print(f"[copy] done -> {dst}")

def split_dataset(images, train_ratio=0.70, seed=42):
    """Split image names into train / val / test, stratified by category.
    The rest after training is divided equally into validation and test.
    """
    cats = [category(f) for f in images]

    # First split: 70% train, 30% rest
    train, rest, _, rest_cats = train_test_split(
        images, cats, train_size=train_ratio, stratify=cats, random_state=seed
    )
    # Second split: the 30% rest -> 15% val, 15% test
    val, test = train_test_split(
        rest, test_size=0.5, stratify=rest_cats, random_state=seed
    )
    return {"train": sorted(train), "val": sorted(val), "test": sorted(test)}

def link_or_copy(src, dst):
    """Put a file into the new folder using a hard link (instant, no extra space).
    If that is not possible, copy it instead.
    """
    if os.path.exists(dst):
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copyfile(src, dst)


def build_yolo_folders(source, out_dir, splits):
    """Create out_dir/<split>/images and labels, plus a <split>.txt file list."""
    for split, files in splits.items():
        for sub in ("images", "labels"):
            os.makedirs(os.path.join(out_dir, split, sub), exist_ok=True)

        for name in files:
            link_or_copy(os.path.join(source, "images", name),
                         os.path.join(out_dir, split, "images", name))
            link_or_copy(os.path.join(source, "labels", label_name(name)),
                         os.path.join(out_dir, split, "labels", label_name(name)))

        with open(os.path.join(out_dir, f"{split}.txt"), "w") as f:
            f.write("\n".join(files))

def write_data_yaml(out_dir):
    """Write data.yaml: tells YOLO where the data is and what the classes are."""
    path = os.path.join(out_dir, "data.yaml")
    with open(path, "w") as f:
        f.write(f"path: {out_dir}\n")
        f.write("train: train/images\n")
        f.write("val: val/images\n")
        f.write("test: test/images\n\n")
        f.write(f"nc: {len(CLASSES)}\n")
        f.write(f"names: {CLASSES}\n")
    return path

def report_split(source, splits):
    """Print images, boxes and the share of boxes per class in each split."""
    print(f"{'split':<6} {'images':>7} {'boxes':>7}  " + "  ".join(f"{c[:7]:>7}" for c in CLASSES))
    for split, files in splits.items():
        boxes = Counter()
        for name in files:
            with open(os.path.join(source, "labels", label_name(name))) as f:
                for line in f:
                    if line.strip():
                        boxes[int(line.split()[0])] += 1
        total = sum(boxes.values())
        shares = "  ".join(f"{100 * boxes[i] / total:6.1f}%" for i in range(len(CLASSES)))
        print(f"{split:<6} {len(files):>7} {total:>7}  {shares}")

def main():
    parser = argparse.ArgumentParser(description="Prepare the garbage dataset for YOLO training.")
    parser.add_argument("--src", required=True, help="Original dataset folder with images/ and labels/")
    parser.add_argument("--work", default="/kaggle/working", help="Writable working directory")
    parser.add_argument("--train-ratio", type=float, default=0.70, help="Share of images for training")
    parser.add_argument("--seed", type=int, default=42, help="Random seed for a reproducible split")
    args = parser.parse_args()

    local_copy = os.path.join(args.work, "GarbageDataset")
    out_dir = os.path.join(args.work, "data")

    copy_dataset(args.src, local_copy)

    images = sorted(os.listdir(os.path.join(local_copy, "images")))
    splits = split_dataset(images, args.train_ratio, args.seed)
    build_yolo_folders(local_copy, out_dir, splits)

    yaml_path = write_data_yaml(out_dir)
    print(f"\n[yaml] {yaml_path}\n" + open(yaml_path).read())

    report_split(local_copy, splits)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train.py
"""Task 2: train and evaluate a YOLOv8 detector on the garbage dataset.

Steps
1. Train YOLOv8 (COCO-pretrained) on the train split, validate on val each epoch.
2. Keep the best checkpoint (highest val fitness) via early stopping.
3. Evaluate best.pt ONCE on the held-out test split.
4. Save overall metrics, a per-class table and the confusion matrix.

Usage
    python train.py --data data/data.yaml                    # final: imgsz 416
    python train.py --data data/data.yaml --imgsz 640 --name yolov8s_baseline   # baseline
    python train.py --data data/data.yaml --aug augmentation.yaml   # settings from Task 1b

The augmentation file is a small YAML of Ultralytics augmentation keys,
for example:  fliplr: 0.5, hsv_v: 0.3, mosaic: 1.0
"""
import argparse
import json
import os

import yaml
from ultralytics import YOLO


# Training settings. Each value is documented with the reason it was chosen.
DEFAULTS = {
    "model": "yolov8s.pt",  # small model: more capacity than 'n' for tiny objects, fits a T4/P100
    "imgsz": 416,           # chosen in Task 1b (416 vs 640 experiment); baseline run used 640
    "epochs": 100,          # upper bound, early stopping usually ends earlier
    "patience": 20,         # stop if val mAP does not improve for 20 epochs
    "batch": 16,            # fits 16 GB GPU memory at 640
    "optimizer": "SGD",     # stable, well tested for fine-tuning YOLO
    "lr0": 0.01,            # initial learning rate (Ultralytics default for SGD)
    "lrf": 0.01,            # final LR = lr0 * lrf (linear decay)
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3,     # slow start so pretrained weights are not destroyed
    "seed": 42,             # same seed as the data split -> reproducible
    "max_det": 400,         # some images contain up to 358 objects (default 300 would cap recall)
}


def parse_args():
    p = argparse.ArgumentParser(description="Train and evaluate YOLOv8 on the garbage dataset.")
    p.add_argument("--data", required=True, help="Path to data.yaml")
    p.add_argument("--aug", default=None, help="Optional YAML with augmentation settings")
    p.add_argument("--project", default="runs", help="Output folder")
    p.add_argument("--name", default="yolov8s_416", help="Run name")
    p.add_argument("--device", default=None, help="'0' for GPU, 'cpu', 'mps'")
    for key, value in DEFAULTS.items():
        p.add_argument(f"--{key}", type=type(value), default=value)
    return p.parse_args()


def load_augmentation(path):
    """Read augmentation settings (Task 1b). Empty dict = Ultralytics defaults."""
    if path is None:
        return {}
    with open(path) as f:
        return yaml.safe_load(f) or {}


def train(args):
    """Fine-tune a COCO-pretrained YOLOv8 model and return the path to best.pt."""
    model = YOLO(args.model)
    model.train(
        data=args.data,
        imgsz=args.imgsz,
        epochs=args.epochs,
        patience=args.patience,
        batch=args.batch,
        optimizer=args.optimizer,
        lr0=args.lr0,
        lrf=args.lrf,
        momentum=args.momentum,
        weight_decay=args.weight_decay,
        warmup_epochs=args.warmup_epochs,
        seed=args.seed,
        deterministic=True,
        pretrained=True,
        project=os.path.abspath(args.project),
        name=args.name,
        exist_ok=True,
        device=args.device,
        max_det=args.max_det,
        plots=True,
        **load_augmentation(args.aug),
    )
    return os.path.join(model.trainer.save_dir, "weights", "best.pt")


def evaluate(weights, args):
    """Evaluate on the TEST split once and save metrics + confusion matrix."""
    model = YOLO(weights)
    metrics = model.val(
        data=args.data,
        split="test",
        imgsz=args.imgsz,
        batch=args.batch,
        project=os.path.abspath(args.project),
        name=f"{args.name}_test",
        exist_ok=True,
        device=args.device,
        max_det=args.max_det,
        plots=True,  # writes confusion_matrix.png and PR curves
    )
    out_dir = str(metrics.save_dir)
    box = metrics.box
    names = model.names

    per_class = []
    for i, cls in enumerate(metrics.ap_class_index):
        p, r, ap50, ap = box.class_result(i)
        per_class.append({
            "class": names[int(cls)],
            "precision": round(float(p), 4),
            "recall": round(float(r), 4),
            "mAP50": round(float(ap50), 4),
            "mAP50-95": round(float(ap), 4),
        })

    summary = {
        "weights": weights,
        "mAP50": round(float(box.map50), 4),
        "mAP50-95": round(float(box.map), 4),
        "precision": round(float(box.mp), 4),
        "recall": round(float(box.mr), 4),
        "per_class": per_class,
    }
    with open(os.path.join(out_dir, "test_metrics.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\nTEST  mAP@0.5={summary['mAP50']:.3f}  mAP@0.5:0.95={summary['mAP50-95']:.3f}  "
          f"P={summary['precision']:.3f}  R={summary['recall']:.3f}")
    print(f"{'class':<14}{'P':>8}{'R':>8}{'mAP50':>8}{'mAP50-95':>10}")
    for row in per_class:
        print(f"{row['class']:<14}{row['precision']:>8.3f}{row['recall']:>8.3f}"
              f"{row['mAP50']:>8.3f}{row['mAP50-95']:>10.3f}")
    print(f"\nSaved to {out_dir}")
    return summary


def main():
    args = parse_args()
    best = train(args)
    evaluate(best, args)


if __name__ == "__main__":
    main()


In [ ]:
# Merge the Kaggle train/valid/test folders into one flat folder (what prepare_data.py expects)
# 64 threads: the /kaggle/input drive is slow for many small files
import glob, os, shutil
from concurrent.futures import ThreadPoolExecutor
yaml_file = glob.glob("/kaggle/input/**/data.yaml", recursive=True)[0]
root = os.path.dirname(yaml_file)
print("dataset root:", root)
dst = "/kaggle/working/GarbageDataset"
jobs = []
for sub in ("images", "labels"):
    os.makedirs(os.path.join(dst, sub), exist_ok=True)
for split in ("train", "valid", "test"):
    for sub in ("images", "labels"):
        d = os.path.join(root, split, sub)
        jobs += [(os.path.join(d, n), os.path.join(dst, sub, n)) for n in os.listdir(d)]
with ThreadPoolExecutor(64) as pool:
    list(pool.map(lambda p: shutil.copyfile(*p), jobs))
print("images:", len(os.listdir(dst + "/images")), "labels:", len(os.listdir(dst + "/labels")))


In [ ]:
!python prepare_data.py --src /kaggle/working/GarbageDataset --work /kaggle/working
!md5sum /kaggle/working/data/test.txt /kaggle/working/data/train.txt

In [ ]:
!python train.py --data /kaggle/working/data/data.yaml --device 0 --project /kaggle/working/runs --imgsz 416 --name yolov8s_416

In [ ]:
# Keep only results in the output (drop the 500 MB of copied images)
!cp /kaggle/working/data/*.txt /kaggle/working/data/data.yaml /kaggle/working/runs/
!rm -rf /kaggle/working/GarbageDataset /kaggle/working/data
!cd /kaggle/working && zip -qr results.zip runs && ls -la /kaggle/working